In [34]:
%load_ext autoreload
%autoreload 2

from radar.components import geometry
from radar.utils.calculate import convert
from radar.utils.typing.enums import FrequencyUnit
from radar.utils.typing.units import Frequency

from radar.components import Element
from radar.utils.calculate import pattern
from radar.utils.typing import (
    PhaseUnit,
    DirectionDomain,
    FigureType,
    AmplitudeDomain,
    Angle,
    AmplitudeUnit,
)

from radar.components.array import Array
import polars as pl

from radar.utils.typing.constants import DataHeader

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [35]:
beam_width = 30
beam_width_tuple = (
    Angle(beam_width, PhaseUnit.DEGREE),
    Angle(beam_width, PhaseUnit.DEGREE),
)

az_bound = 90
el_bound = 90
az_bound_tuple = (Angle(-az_bound, PhaseUnit.DEGREE), Angle(az_bound, PhaseUnit.DEGREE))
el_bound_tuple = (Angle(-el_bound, PhaseUnit.DEGREE), Angle(el_bound, PhaseUnit.DEGREE))


element_pattern = pattern.Isotropic()
freq = Frequency(1, FrequencyUnit.GIGAHERTZ)
antenna_element = Element(element_pattern, az_bound_tuple, el_bound_tuple, freq, 1)

cf = Frequency(1, FrequencyUnit.GIGAHERTZ)
distance = convert.cf_to_min_dist(cf)
array_geometry = geometry.Grid(5, 5, distance)

arr = Array(antenna_element, array_geometry)

arr.plot.beam(
    DirectionDomain.ANGLE,
    PhaseUnit.DEGREE,
    AmplitudeDomain.AntennaFactor,
    AmplitudeUnit.DECIBEL,
    FigureType.SURFACE,
    Frequency(1, FrequencyUnit.GIGAHERTZ),
)

In [36]:
# import array

# from radar.optimiser.biology import Organism

# def calculate_genetic_health(org: Organism) -> float:
#     score = 0.0

#     for chromosome in org.chromosomes:
#         # Access the underlying Polars DataFrame
#         # df = chromosome.df

#         az_bound = 90
#         el_bound = 90
#         az_bound_tuple = (Angle(-az_bound, PhaseUnit.DEGREE), Angle(az_bound, PhaseUnit.DEGREE))
#         el_bound_tuple = (Angle(-el_bound, PhaseUnit.DEGREE), Angle(el_bound, PhaseUnit.DEGREE))


#         element_pattern = pattern.Isotropic()
#         freq = Frequency(1, FrequencyUnit.GIGAHERTZ)
#         antenna_element = Element(element_pattern, az_bound_tuple, el_bound_tuple, freq, 1)

#         cf = Frequency(1, FrequencyUnit.GIGAHERTZ)
#         distance = convert.cf_to_min_dist(cf)
#         array_geometry = geometry.Grid(5, 5, distance)
#         # 1. Start with the original DataFrame
#         df = array_geometry.df

#         # print(sum(df[DataHeader.GEOM_AMP_GAIN_DB]))
#         # 2. Chain the operations together so the data flows seamlessly
#         updated_df = (
#             df.with_columns(
#                 chromosome.df.to_series().alias(DataHeader.GEOM_AMP_GAIN_DB)
#             )
#             .drop(DataHeader.GEOM_AMP_GAIN_LIN)
#         )

#         array_geometry.gains = updated_df

#         arr = Array(antenna_element, array_geometry)

#         return arr.statistic.std(cf)


#     return score

In [37]:
# import array
# from typing import cast
# import polars as pl

# from radar.optimiser.biology import Organism
# from IPython.display import clear_output

# def calculate_genetic_health2(org: Organism, generation : int | None = None, plot = False) -> float:

#     x = org._chromosomes[0]
#     y = org._chromosomes[1]

#     az_bound = 45
#     el_bound = 45
#     az_bound_tuple = (Angle(-az_bound, PhaseUnit.DEGREE), Angle(az_bound, PhaseUnit.DEGREE))
#     el_bound_tuple = (Angle(-el_bound, PhaseUnit.DEGREE), Angle(el_bound, PhaseUnit.DEGREE))


#     element_pattern = pattern.Isotropic()
#     freq = Frequency(1, FrequencyUnit.GIGAHERTZ)
#     antenna_element = Element(element_pattern, az_bound_tuple, el_bound_tuple, freq, 1)

#     cf = Frequency(1, FrequencyUnit.GIGAHERTZ)
#     array_geometry = geometry.CustomGeometry(x.df.to_numpy().ravel(), y.df.to_numpy().ravel())

#     arr = Array(antenna_element, array_geometry)

#     beam = arr.beam_pattern(cf, None)
#     # Fixed: Added parentheses around conditions and used native Polars .abs()
#     ave = (
#         beam.filter(
#             (pl.col(DataHeader.AZIMUTH_DEG).abs() > 20) &
#             (pl.col(DataHeader.ELEVATION_DEG).abs() > 20)
#         )
#         .select(DataHeader.ANTENNA_FACTOR_DB)
#         .to_series()
#         .mean()
#     )

#     ave_beam = (
#         beam.filter(
#             (pl.col(DataHeader.AZIMUTH_DEG).abs() < 20) &
#             (pl.col(DataHeader.ELEVATION_DEG).abs() < 20)
#         )
#         .select(DataHeader.ANTENNA_FACTOR_DB)
#         .to_series()
#         .mean()
#     )
#     ave = cast(float, ave)
#     ave_beam = cast(float, ave_beam)
#     std = arr.statistic.std(cf)

#     if plot:
#         clear_output(wait=True)
#         arr.plot.beam(
#             DirectionDomain.ANGLE,
#             PhaseUnit.DEGREE,
#             AmplitudeDomain.AntennaFactor,
#             AmplitudeUnit.DECIBEL,
#             FigureType.SURFACE,
#             Frequency(1, FrequencyUnit.GIGAHERTZ),
#             None,
#             f"tmp/beam_{generation}"
#         )

#         print(f"-----  {ave} {ave_beam} {std}")


#     # lower is better


#     return std + 4*(ave - ave_beam) # arr.statistic.ave(cf)


In [48]:
import array
from typing import Literal, cast
import polars as pl

from radar.optimiser.biology import Organism
from IPython.display import clear_output
from radar.utils.typing.enums import BoundType


def calculate_genetic_health3(
    org: Organism, generation: int | None = None, plot=False
) -> dict:

    x = org._chromosomes[0]
    y = org._chromosomes[1]
    g = org._chromosomes[2]

    # print(g.df)

    az_bound = 90
    el_bound = 90
    az_bound_tuple = (
        Angle(-az_bound, PhaseUnit.DEGREE),
        Angle(az_bound, PhaseUnit.DEGREE),
    )
    el_bound_tuple = (
        Angle(-el_bound, PhaseUnit.DEGREE),
        Angle(el_bound, PhaseUnit.DEGREE),
    )

    element_pattern = pattern.Isotropic()
    freq = Frequency(1, FrequencyUnit.GIGAHERTZ)
    antenna_element = Element(element_pattern, az_bound_tuple, el_bound_tuple, freq, 1)

    cf = Frequency(1, FrequencyUnit.GIGAHERTZ)
    array_geometry = geometry.CustomGeometry(
        x.df.to_numpy().ravel(), y.df.to_numpy().ravel()
    )

    # Geometry
    df = array_geometry.df
    updated_df = df.with_columns(
        g.df.to_series().alias(DataHeader.GEOM_AMP_GAIN_DB)
    ).drop(DataHeader.GEOM_AMP_GAIN_LIN)
    array_geometry.gains = updated_df

    arr = Array(antenna_element, array_geometry)

    beam = arr.beam_pattern(cf, None)
    beam2 = beam.clone()

    beam2 = beam2.with_columns(
        pl.when(
            (pl.col(DataHeader.AZIMUTH_DEG).abs() > 10)
            & (pl.col(DataHeader.ELEVATION_DEG).abs() > 10)
        )
        .then(1)  # If condition is True -> 0
        .otherwise(0.01)  # If condition is False -> -20
        .alias(DataHeader.BEAM_GAIN_LINEAR)
    )

    average_diff = (
        beam2.select(
            (
                pl.col(DataHeader.BEAM_GAIN_LINEAR)
                - pl.lit(beam[DataHeader.BEAM_GAIN_LINEAR])
            )
            .abs()
            .mean()
        ).item()  # Extracts the single value from the resulting 1x1 DataFrame
    )

    filters = [
        (DataHeader.AZIMUTH_DEG, -20.0, 20, BoundType.EXCLUSIVE),
        (DataHeader.ELEVATION_DEG, -20, 20, BoundType.EXCLUSIVE),
    ]
    ave_db = arr.statistic.mean(DataHeader.ANTENNA_FACTOR_DB, freq, filters, None, "OR")
    std_db = arr.statistic.std(DataHeader.ANTENNA_FACTOR_DB, freq, filters, None, "OR")
    max_db = arr.statistic.max(DataHeader.ANTENNA_FACTOR_DB, freq, filters, None, "OR")
    min_db = arr.statistic.max(DataHeader.ANTENNA_FACTOR_DB, freq, filters, None, "OR")

    ave_prop = 200 * (10 ** (ave_db / 10.0))
    std_dev_prop = 40 * (10 ** (std_db / 10.0))
    diff_prop = 8 * average_diff
    max_prop = 4 * (10 ** (max_db / 10))

    diff = (10 ** ((max_db-min_db) / 10))
    fitness = (ave_prop + std_dev_prop + diff_prop) * max_prop * diff

    return {
        "fitness": fitness,
        "average_db": ave_db,
        "standard_deviation_db": std_db,
        "ave_diff_lin": average_diff,
        "ave_prop": ave_prop,
        "std_dev_prop": std_dev_prop,
        "diff_prop": diff_prop,
        "max_db": max_db,
        "max_prop": max_prop,
    }  # arr.statistic.ave(cf)

In [45]:
# 1. Initialize your population
from radar.optimiser.biology import Population

pop = Population(
    5,
    400,
    [DataHeader.X_POS_M, DataHeader.Y_POS_M, DataHeader.GEOM_AMP_GAIN_DB],
    25,
    [-0.5, -0.5, -3],
    [0.5, 0.5, 0],
    0.60,
)
if "generation" in locals():
    del generation

FigureWidget({
    'data': [],
    'layout': {'legend': {'itemclick': 'toggle',
                          'itemdoubleclick': 'toggleothers',
                          'orientation': 'h',
                          'x': 1,
                          'xanchor': 'right',
                          'y': 1.02,
                          'yanchor': 'bottom'},
               'showlegend': True,
               'template': '...',
               'title': {'text': 'Live Dynamic Plot'}}
})

In [33]:
best = pop._elite_community._organisms[0]
best.save("tmp5.csv")

In [46]:
pop.seed(Organism.load("tmp5.csv", [-0.5, -0.5, -10], [0.5, 0.5, 0]))

In [ ]:
import datetime

import logging

# Mute the kaleido logger specifically
logging.getLogger("kaleido").setLevel(logging.ERROR)
# # 2. Define how many generations you want to evolve
num_generations = 500000

print("Starting optimization loototal_diffp...\n")

start = 1
if "generation" in locals():
    start = generation

for generation in range(start, num_generations + 1):
    # 3. Propagate the population to the next generation
    # Pass the function name without parentheses!

    if generation % 15:
        pop.propogate(
            fitness_fn=calculate_genetic_health3, elite=False, generation=generation
        )
    else:
        pop.propogate(
            fitness_fn=calculate_genetic_health3, elite=True, generation=generation
        )
        best = pop._elite_community._organisms[0]
        best.save("tmp7.csv")
        # tmp = pop._elite_community._organisms[0]
        # calculate_genetic_health3(tmp, generation, True)

    # print(f"Generation {generation}/{num_generations}:")

Starting optimization loototal_diffp...



KeyboardInterrupt: 

In [44]:
x = pop._elite_community._organisms[0]._chromosomes[0]
y = pop._elite_community._organisms[0]._chromosomes[1]
c = pop._elite_community._organisms[0]._chromosomes[2]

az_bound = 90
el_bound = 90
az_bound_tuple = (Angle(-az_bound, PhaseUnit.DEGREE), Angle(az_bound, PhaseUnit.DEGREE))
el_bound_tuple = (Angle(-el_bound, PhaseUnit.DEGREE), Angle(el_bound, PhaseUnit.DEGREE))


element_pattern = pattern.Isotropic()
freq = Frequency(1, FrequencyUnit.GIGAHERTZ)
antenna_element = Element(element_pattern, az_bound_tuple, el_bound_tuple, freq, 1)

cf = Frequency(1, FrequencyUnit.GIGAHERTZ)

array_geometry = geometry.CustomGeometry(
    x.df.to_numpy().ravel(), y.df.to_numpy().ravel()
)
df = array_geometry.df
updated_df = df.with_columns(c.df.to_series().alias(DataHeader.GEOM_AMP_GAIN_DB)).drop(
    DataHeader.GEOM_AMP_GAIN_LIN
)
array_geometry.gains = updated_df

arr = Array(antenna_element, array_geometry)

arr.plot.beam(
    DirectionDomain.ANGLE,
    PhaseUnit.DEGREE,
    AmplitudeDomain.AntennaFactor,
    AmplitudeUnit.DECIBEL,
    FigureType.SURFACE,
    Frequency(1, FrequencyUnit.GIGAHERTZ),
)

arr.plot.geometry()